In [105]:
import boto3
import pandas as pd
import streamlit as st
import awswrangler as wr
import plotly.express as px
from datetime import datetime

In [106]:
session = boto3.Session(profile_name='hiv-project')
s3 = session.client('s3')
bucket_name = 'hiv-data-022784797781'

In [107]:
# Which combination of drug classes were patients switched to after TCE?

In [108]:
sql = """
SELECT * 
FROM hivdb_tce.v_new_regimen_class_counts
ORDER BY tce_count DESC;
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

In [109]:
threshold = 7

df_plot = df.copy()
df_plot.loc[df_plot['tce_count'] < threshold, 'class_signature'] = 'Other'
df_plot = df_plot.groupby('class_signature', as_index=False)['tce_count'].sum()

df_plot = df_plot.sort_values('tce_count', ascending=False)
other_row = df_plot[df_plot['class_signature'] == 'Other']
df_plot = pd.concat([df_plot[df_plot['class_signature'] != 'Other'], other_row])

colors = px.colors.qualitative.Prism
color_map = {sig: colors[i % len(colors)]
             for i, sig in enumerate(df_plot[df_plot['class_signature'] != 'Other']['class_signature'])}
color_map['Other'] = '#808080'

legend_order = list(df_plot[df_plot['class_signature'] != 'Other']['class_signature']) + ['Other']

fig = px.pie(
    df_plot,
    names="class_signature",
    values="tce_count",
    hole=0.4,
    color="class_signature",
    color_discrete_map=color_map,
    category_orders={"class_signature": legend_order}
)

fig.show()
fig.write_image("drug_class_combinations.png")

In [110]:
# TCE Frequency by Year

In [111]:
sql = """
select * from hivdb_tce.tce_by_year order by baseline_year
"""

df = wr.athena.read_sql_query(
    sql=sql,
    database="hivdb_tce",
    boto3_session=session,
    s3_output="s3://hiv-data-022784797781/athena-results/"
)

df["baseline_year"] = df["baseline_year"].astype(int)
df = df.sort_values("baseline_year")

fig = px.line(
    df,
    x="baseline_year",
    y="tce_count",
    markers=True
)

fig.update_layout(
    title="TCE Frequency by Year",
    xaxis_title="Baseline Year",
    yaxis_title="Number of Treatment Change Events"
)

fig.show()
fig.write_image("tce_freq_by_year.png")